# Receipt VLM — training on **Vertex AI Colab Enterprise** (GCP)

Runs the 3-phase curriculum on a **Colab Enterprise runtime** with a GPU (T4 / L4 / A100).
Like Kaggle/Colab the runtime is **ephemeral**: `/content` is wiped when the runtime is
deleted or hits its idle / max-runtime limit.

**So every output is written *straight into a Cloud Storage (GCS) bucket*.** The bucket is
mounted with `gcsfuse` (cell 2) and `checkpoint_dir` points at the mount, so each
`phase*_best.pt` lands in GCS the *instant* the trainer saves it — on every validation
improvement, not just at phase boundaries. A teardown at any moment loses nothing already saved.

> **Workbench vs Colab Enterprise** — both are "Vertex AI notebooks", but:
>
> | | Workbench | **Colab Enterprise** |
> |---|---|---|
> | What it is | a full JupyterLab **VM** you own | a managed **runtime** from a *runtime template* |
> | Home disk | **persistent** (`/home/jupyter` survives stop/start) | **ephemeral** (`/content`, lost on teardown) |
> | Lifecycle | you Stop/Start the VM | auto **idle shutdown** (~180 min) + max runtime |
> | GPU choice | per-instance at create time | fixed in the **runtime template** |
> | Billing | per hour while *running* | per hour while the *runtime* is up |
> | Auth | runs as a service account (`gsutil` pre-auth) | same — runs as the template's service account |
>
> Because the disk is ephemeral *and* the runtime can idle-shutdown mid-run, this notebook
> writes checkpoints **directly to the bucket** via `gcsfuse` instead of syncing after each phase.

## One-time setup (before running)
1. **Create a runtime template:** Vertex AI → *Colab Enterprise* → *Runtime templates* → *Create*.
   - Machine: `n1-standard-8` + **NVIDIA T4** (cheapest), or `g2-standard-12` + **L4**, or A100.
   - Disk: 100 GB+ (HF cache + CORD live on local disk; only checkpoints go to the bucket).
   - Idle shutdown: raise it (e.g. 60–180 min) so a quiet startup download doesn't kill the runtime.
   - Service account: the Compute default SA, or a custom one (see permissions below).
2. **Build the bundle on your PC** and upload it to a bucket:
   ```powershell
   cd dev_ocr\vlm_training
   .venv\Scripts\python scripts\zip_selfcontained_colab.py
   gsutil cp colab_upload\receipt_vlm_colab_bundle.zip gs://YOUR_BUCKET/receipt_vlm/
   ```
3. **Permissions:** the runtime's service account needs **Storage Object Admin** on
   `gs://YOUR_BUCKET` (read the bundle + mount/write checkpoints):
   ```powershell
   gsutil iam ch serviceAccount:SA_EMAIL:roles/storage.objectAdmin gs://YOUR_BUCKET
   ```
4. **Import this notebook** into Colab Enterprise (Notebooks → *Import* → upload the `.ipynb`),
   connect it to a runtime built from your template, edit **cell 0** (`GCS_BUCKET`), then *Run all*.

> Full guide: [`COLAB_ENTERPRISE.md`](../COLAB_ENTERPRISE.md).

## 0. Configuration — edit then run

In [ ]:
# --- GCS (no gs:// prefix on the bucket name) ---
GCS_BUCKET = "YOUR_BUCKET"          # bucket holding the bundle + checkpoints
GCS_PREFIX = "receipt_vlm"          # folder within the bucket
BUNDLE_NAME = "receipt_vlm_colab_bundle.zip"

# --- which phases to run (set False to skip; skipped phases need their checkpoint) ---
RUN_PHASE_1 = True
RUN_PHASE_2 = True
RUN_PHASE_3 = True
RUN_EXPORT  = True

FORCE_SMALL_BATCH = False  # set True on a single small GPU if you hit CUDA OOM (batch 8 -> 4)

GCS_BASE = f"gs://{GCS_BUCKET}/{GCS_PREFIX}"
assert GCS_BUCKET != "YOUR_BUCKET", "Set GCS_BUCKET to your real bucket name first."
print("Bundle :", f"{GCS_BASE}/{BUNDLE_NAME}")
print("Outputs land in :", f"{GCS_BASE}/checkpoints  (via gcsfuse mount, cell 2)")

## 1. GPU & identity check
Confirms the runtime has a GPU and that you can reach the bucket as the runtime's service
account. If the bucket check fails, grant the SA `roles/storage.objectAdmin` (cell 0 setup).

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU on this runtime — rebuild it from a GPU runtime template"
print("GPU :", torch.cuda.get_device_name(0))

# who am I running as, and can I see the bucket?
acct = subprocess.run(["gcloud", "config", "get-value", "account"],
                      capture_output=True, text=True).stdout.strip()
print("Identity :", acct or "(default runtime service account)")
rc = subprocess.call(["gsutil", "ls", f"gs://{GCS_BUCKET}/{GCS_PREFIX}/"])
assert rc == 0, "Cannot list the bucket — grant the runtime SA roles/storage.objectAdmin on it"

## 2. Mount the bucket with gcsfuse (outputs go straight to GCS)
Mounts `gs://YOUR_BUCKET` at `/content/gcs`. `CKPT_DIR` lives inside the mount, so every
checkpoint the trainer writes is uploaded to the bucket on save — nothing is lost on a teardown.

> Only *checkpoints* go on the mount. The HF cache and the CORD dataset stay on the fast local
> disk (they're re-downloaded on a fresh runtime — they're inputs, not outputs).

In [ ]:
import os, shutil, subprocess
from pathlib import Path

MOUNT = Path("/content/gcs")
MOUNT.mkdir(parents=True, exist_ok=True)

# Install gcsfuse if the runtime image doesn't already ship it.
if not shutil.which("gcsfuse"):
    print("Installing gcsfuse ...", flush=True)
    subprocess.check_call(r"""
export GCSFUSE_REPO=gcsfuse-$(lsb_release -c -s)
echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt $GCSFUSE_REPO main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list >/dev/null
curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo gpg --dearmor -o /usr/share/keyrings/cloud.google.gpg
sudo apt-get update -qq && sudo apt-get install -y -qq gcsfuse
""", shell=True, executable="/bin/bash")

# Mount (idempotent: skip if already mounted). gcsfuse authenticates via the runtime SA (ADC).
if not os.path.ismount(MOUNT):
    subprocess.check_call(["gcsfuse", "--implicit-dirs", GCS_BUCKET, str(MOUNT)])
assert os.path.ismount(MOUNT), "gcsfuse mount failed"

CKPT_DIR = MOUNT / GCS_PREFIX / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
# Prove writes reach the bucket.
(CKPT_DIR / ".write_test").write_text("ok"); (CKPT_DIR / ".write_test").unlink()
print("Mounted gs://%s -> %s" % (GCS_BUCKET, MOUNT))
print("Checkpoints write directly to:", CKPT_DIR)

## 3. Get the code from your GCS bucket
Downloads the bundle with `gsutil` and extracts it under `/content` (the runtime's local disk).
The *code* is disposable — re-run this on a fresh runtime; your *outputs* are already safe in the bucket.

In [ ]:
import glob, os, subprocess, zipfile
from pathlib import Path

ROOT = Path("/content/receipt_vlm")
WORK = ROOT / "repo"
LOCAL_ZIP = ROOT / BUNDLE_NAME

def materialize():
    WORK.mkdir(parents=True, exist_ok=True)
    print("Downloading", f"{GCS_BASE}/{BUNDLE_NAME}", "...")
    subprocess.check_call(["gsutil", "cp", f"{GCS_BASE}/{BUNDLE_NAME}", str(LOCAL_ZIP)])
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(WORK)

if not list(WORK.glob("**/vlm_training/scripts/train.py")):
    materialize()

hits = glob.glob(str(WORK / "**/vlm_training/scripts/train.py"), recursive=True)
assert hits, "train.py not found after materialize — wrong bucket/path?"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]   # .../dev_ocr/vlm_training
DEV_OCR = TRAIN_PKG.parent                        # .../dev_ocr
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

## 4. Install dependencies (~2-3 min)

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")

## 5. Point configs at the bucket mount
`checkpoint_dir` is set to the gcsfuse mount (cell 2), so the trainer writes every
`phase*_best.pt` straight to GCS. The real-data dirs point at the extracted bundle on local disk.

In [ ]:
import yaml
from pathlib import Path

cfg_path = TRAIN_PKG / "configs" / "colab_paths.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
cfg["checkpoint_dir"] = str(CKPT_DIR)          # <- the gcsfuse mount: writes go to the bucket
cfg["log_every"] = 25
if FORCE_SMALL_BATCH:
    cfg["batch_size"] = 4
cfg.setdefault("data", {})
cfg["data"]["real_images_dir"] = str(DEV_OCR / "data" / "raw" / "images_tickets_caisse")
cfg["data"]["real_labels_dir"] = str(TRAIN_PKG / "data" / "real_labels")
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
print(cfg_path.read_text())

## 5b. Checkpoints already in the bucket
Because `CKPT_DIR` *is* the bucket, finished phases are already here on a fresh runtime — no
restore step needed. List them to decide which phases to skip (set `RUN_PHASE_1/2 = False` in cell 0).

In [ ]:
found = []
for name in ("phase1_best.pt", "phase2_best.pt", "phase3_best.pt"):
    p = CKPT_DIR / name
    if p.is_file():
        found.append(f"{name} ({p.stat().st_size/1e6:.0f} MB)")
print("In bucket:", found if found else "none yet — this is a fresh run")

## 6. Train phases 1 -> 2 -> 3
Live timestamp/gap heartbeat. Each `phase*_best.pt` is written **directly to the bucket** on every
validation improvement, so an idle shutdown or disconnect loses nothing already saved. ~3-4 h on T4
(the silent startup downloads CLIP+SmolLM2 then CORD).

In [ ]:
import subprocess, sys, os, time, datetime

p1 = str(CKPT_DIR / "phase1_best.pt")
p2 = str(CKPT_DIR / "phase2_best.pt")
p3 = str(CKPT_DIR / "phase3_best.pt")

def run_train(config, resume=None):
    cmd = [sys.executable, "-u", "scripts/train.py", "--config", config]
    if resume:
        cmd += ["--resume", resume]
    print("\n>>", " ".join(cmd), flush=True)
    print("   (startup is silent for a few min: downloading CLIP+SmolLM2, then CORD)", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    start = last = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        now = time.time()
        gap = now - last; last = now
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{config} failed (exit {proc.returncode})")
    print(f"-- {config} done in {int(time.time()-start)}s (checkpoint already in the bucket)", flush=True)

# Skipping a phase requires its checkpoint already in the bucket (see cell 5b).
if not RUN_PHASE_1 and (RUN_PHASE_2 or RUN_PHASE_3):
    assert os.path.exists(p1), f"{p1} missing in bucket -- set RUN_PHASE_1=True"
if not RUN_PHASE_2 and RUN_PHASE_3:
    assert os.path.exists(p2), f"{p2} missing in bucket -- set RUN_PHASE_2=True"

if RUN_PHASE_1:
    run_train("configs/phase1_colab.yaml")
if RUN_PHASE_2:
    run_train("configs/phase2_colab.yaml", resume=p1)
if RUN_PHASE_3:
    run_train("configs/phase3_colab.yaml", resume=p2)
print("Training done")

## 7. Export merged inference checkpoint (written to the bucket)

In [ ]:
MERGED = str(CKPT_DIR / "receipt_vlm_500m_merged.pt")
if RUN_EXPORT:
    subprocess.check_call([sys.executable, "scripts/export_checkpoint.py",
                           "--checkpoint", p3, "--output", MERGED])
    print("Merged ->", MERGED, "(in the bucket)")
else:
    print("Export skipped")

## 8. Verify outputs in the bucket
Everything below is already durable in `gs://YOUR_BUCKET`. Download the merged model with
`gsutil cp gs://YOUR_BUCKET/receipt_vlm/checkpoints/receipt_vlm_500m_merged.pt .` on your PC.

In [ ]:
from pathlib import Path

print("Outputs in", f"{GCS_BASE}/checkpoints :")
for p in sorted(CKPT_DIR.glob("*")):
    if p.is_file():
        print(f"  {p.name:34} {p.stat().st_size/1e6:8.1f} MB")
print("\nAll of the above live in the bucket already.")
print("Remember to DELETE this runtime (Colab Enterprise -> Runtimes) to stop billing.")

## 9. (optional) Sanity check on one photo

In [ ]:
import json, os
from pathlib import Path

photos = sorted((DEV_OCR / "data" / "raw" / "images_tickets_caisse").glob("*.jpg"))
if photos and Path(MERGED).is_file():
    os.environ.update({
        "RECEIPT_OCR_BACKEND": "vlm",
        "RECEIPT_VLM_MODEL": "receipt-vlm-500m",
        "RECEIPT_VLM_MODE": "json",
        "RECEIPT_VLM_MODEL_PATH": MERGED,
    })
    from receipt_ocr import extract_receipt
    print(json.dumps(extract_receipt(str(photos[0])), indent=2, ensure_ascii=False)[:1200])
else:
    print("Need merged checkpoint + at least one photo")